# 04 · Demo: chat with FairRent Akure

Pipeline: **message → Gemma extracts fees → fee engine checks them against survey data → Gemma writes the reply → safeguard**.

The safeguard checks every number in the model's reply against the engine result. If any number is not in the result,
the demo shows the template reply instead, so tenants never see an invented figure.

**Runs on:** Colab T4 GPU. Needs the LoRA adapter from notebook 02.

In [ ]:
# --- Setup: get the project code (works in Colab and locally) ---
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/fairrent-akure.git"  # <-- change to your repo

import os, sys
if os.path.exists("../src/fairrent"):          # running locally from notebooks/
    os.chdir("..")
elif not os.path.exists("src/fairrent"):       # running in Colab
    folder = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(folder):
        os.system(f"git clone -q {REPO_URL}")
    os.chdir(folder)
sys.path.insert(0, "src")
print("Working in:", os.getcwd())

In [ ]:
!pip install -q "transformers>=4.50" "peft>=0.13" accelerate gradio

In [ ]:
import pandas as pd
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("Using existing Hugging Face login.", e)

from fairrent.engine import FeeEngine
from fairrent.llm import DEFAULT_MODEL_ID, FairRentAssistant, load_model

ADAPTER_PATH = "fairrent-gemma3-1b-lora"
if not os.path.exists(ADAPTER_PATH):
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER_PATH = "/content/drive/MyDrive/fairrent/fairrent-gemma3-1b-lora"

model, tokenizer = load_model(DEFAULT_MODEL_ID, ADAPTER_PATH)
engine = FeeEngine(pd.read_csv("data/tenants_clean.csv", keep_default_na=False, na_values=[""]))
assistant = FairRentAssistant(model, tokenizer, engine)

In [ ]:
for msg in [
    "Agent wan collect money for selfcon for southgate. Rent na 280k, agency 56k, caution 50k, development fee 40k. E make sense?",
    "How much is a single room around North Gate?",
    "Ẹ n lẹ́. Mo fẹ́ gba self-contain ní Apatapiti. Rent jẹ́ 200k, agency fee jẹ́ 20k. Ṣé ó dára?",
    "Who won the match yesterday?",
]:
    out = assistant.chat(msg)
    print("USER:", msg)
    print("FAIRRENT:", out["reply"])
    print(f"(model reply grounded: {out['model_reply_grounded']}, fallback used: {out['used_fallback']})")
    print("-" * 80)

## Interactive demo (share the public link for user testing)

In [ ]:
import gradio as gr

def respond(message, history):
    return assistant.chat(message)["reply"]

gr.ChatInterface(
    respond, title="FairRent Akure",
    description="Paste your rent and fee breakdown (English, Pidgin or Yoruba). Guidance from Akure tenant survey data, not legal advice.",
    examples=["Selfcon for South Gate. Rent 250k, agency 50k, agreement 25k, caution 30k. Is this fair?",
              "How much is self-contain for northgate?"],
).launch(share=True)